In [1]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.sparse.linalg import eigsh, ArpackNoConvergence
from scipy.sparse import diags
from scipy.interpolate import interp1d
from scipy.integrate import quad
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sys

# -------------------------------
# Función auxiliar de depuración
def debug_log(message):
    print(message, flush=True)
    with open("debug_log.txt", "a") as f:
        f.write(message + "\n")

# ---------------------------------------------
# 1. Parámetros Físicos Optimizados
# ---------------------------------------------
D0 = 2e-9           # Coeficiente de difusión [m²/s]
R = 1e-6            # Radio del cilindro [m]
z_c = 10e-9         # Altura crítica [m]
lambda_param = 0.2  # Parámetro de decaimiento
delta = 1e-9        # Ancho de transición
n_modes_radial = 8  # Número de modos radiales
m_max = 10          # Número de modos axiales
N_z = 600           # Número de nodos en z
alpha_grid = 8.0    # Parámetro de malla
L = 2e-8            # Longitud total en z (malla de 0 a 2e-8)

# ---------------------------------------------
# 2. Función de Difusión Global
# ---------------------------------------------
def D(z_val):
    """Perfil de difusión completo."""
    S = 0.5 * (1 + np.tanh((z_val - z_c) / delta))
    return D0 * (np.exp(-lambda_param * z_val) * (1 - S) + S)

# ---------------------------------------------
# 3. Generación de la Malla en z
# ---------------------------------------------
j_vals = np.linspace(0, 1, N_z)
z = z_c * (1 + np.sinh(alpha_grid * (j_vals - 0.5)) / np.sinh(0.5 * alpha_grid))
# La malla se extiende de aproximadamente 0 a L (L = 2e-8 m)
D_values = D(z)
debug_log(f"Malla en z: min = {np.min(z):.2e}, max = {np.max(z):.2e}")

# ---------------------------------------------
# 4. Solución Radial
# ---------------------------------------------
alpha_zeros = jn_zeros(0, n_modes_radial)
alpha_n = alpha_zeros / R
rho_plot = np.linspace(0.1 * R, 0.9 * R, 80)
J0_table = jv(0, alpha_n[:, None] * rho_plot)
debug_log(f"alpha_n: {alpha_n}")

# ---------------------------------------------
# 5. Solución Axial con Manejo de Errores
# ---------------------------------------------
def build_system(n):
    try:
        alpha = alpha_n[n]
        h = np.diff(z)
        N = len(z)
        main_diag = np.zeros(N)
        upper_diag = np.zeros(N - 1)
        lower_diag = np.zeros(N - 1)
        for j in range(1, N - 1):
            h_prev = h[j - 1]
            h_next = h[j]
            D_prev = 2 * D_values[j] * D_values[j - 1] / (D_values[j] + D_values[j - 1] + 1e-30)
            D_next = 2 * D_values[j] * D_values[j + 1] / (D_values[j] + D_values[j + 1] + 1e-30)
            h_avg = 0.5 * (h_prev + h_next)
            main_diag[j] = D_prev / (h_prev * h_avg) + D_next / (h_next * h_avg) + alpha**2 * D_values[j]
            upper_diag[j] = -D_next / (h_next * h_avg)
            lower_diag[j - 1] = -D_prev / (h_prev * h_avg)
        main_diag[0] = D_values[0] / h[0] + 1e-10 * D0
        upper_diag[0] = -D_values[0] / h[0]
        main_diag[-1] = 1.0
        A = diags([main_diag, upper_diag, lower_diag], [0, 1, -1])
        eigenvalues, eigenvectors = eigsh(A, k=m_max, which='SA', tol=1e-6, maxiter=15000)
        debug_log(f"Modo radial {n}: eigenvalues: {eigenvalues}")
        return (n, (eigenvalues, eigenvectors))
    except ArpackNoConvergence:
        debug_log(f"Advertencia: Modo radial {n} no convergió. Usando ceros.")
        return (n, (np.zeros(m_max), np.zeros((N, m_max))))

results = Parallel(n_jobs=4, verbose=1)(delayed(build_system)(n) for n in range(n_modes_radial))
eigenvalues_dict = {}
eigenvectors_dict = {}
for n, (evals, evecs) in sorted(results, key=lambda x: x[0]):
    eigenvalues_dict[n] = evals
    eigenvectors_dict[n] = evecs

# ---------------------------------------------
# 6. Visualización de Modos (Radiales y Axiales)
# ---------------------------------------------
def safe_divide(a, b):
    return np.divide(a, b, out=np.zeros_like(a), where=b != 0)

plt.figure(figsize=(10, 6))
for n in range(4):
    mode = J0_table[n]
    plt.plot(rho_plot / R, safe_divide(mode, np.max(np.abs(mode))),
             label=f'n={n+1}', linewidth=1.5)
plt.xlabel(r'$\rho/R$', fontsize=12)
plt.ylabel('Amplitud Normalizada', fontsize=12)
plt.title('Modos Radiales de Bessel', fontsize=14)
plt.legend()
plt.grid(alpha=0.4)
plt.savefig('modos_radiales.pdf', bbox_inches='tight')
plt.close()

fig, axs = plt.subplots(2, 2, figsize=(12, 8))
for idx, (n, m) in enumerate([(0, 0), (0, 1), (1, 0), (1, 1)]):
    ax = axs[idx // 2, idx % 2]
    mode = eigenvectors_dict[n][:, m]
    norm_mode = safe_divide(mode, np.max(np.abs(mode)))
    ax.plot(z * 1e9, norm_mode, color='darkred')
    ax.set_xlabel('z (nm)', fontsize=10)
    ax.set_title(f'Modo (n={n}, m={m})', fontsize=12)
    ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('modos_axiales.pdf', bbox_inches='tight')
plt.close()

# ---------------------------------------------
# 7. Cálculo de Coeficientes A_nm con Depuración
# ---------------------------------------------
Z_interps = {}
for n in range(n_modes_radial):
    Z_interps[n] = [interp1d(z, eigenvectors_dict[n][:, m],
                             bounds_error=False, fill_value=0.0)
                    for m in range(m_max)]

def compute_A_nm(n):
    Z_norms = np.array([quad(lambda z_val: (Z_interps[n][m](z_val)**2) / D(z_val),
                              z[0], z[-1], limit=300)[0]
                        for m in range(m_max)])
    Z_m_z0 = np.array([Z_interps[n][m](15e-9) for m in range(m_max)])
    J0_rho0 = jv(0, alpha_n[n] * 0.75e-6)
    valid = (Z_norms > 1e-25) & (np.abs(Z_m_z0) > 1e-12)
    debug_log(f"A_nm for n={n}: Z_norms = {Z_norms}, Z_m_z0 = {Z_m_z0}, valid: {valid}")
    return (J0_rho0 * Z_m_z0 * valid) / (2 * np.pi * 0.75e-6 * (Z_norms + 1e-30))

A_nm = Parallel(n_jobs=4)(delayed(compute_A_nm)(n) for n in range(n_modes_radial))
for n in range(n_modes_radial):
    debug_log(f"A_nm[{n}] = {A_nm[n]}")

# ---------------------------------------------
# 8. Función de Correlación G(t) con Depuración y Normalización
# ---------------------------------------------
def compute_G_debug(t, N_quad=12):
    debug_log(f"Calculando G(t) para t = {t:.1e}")
    # Usamos ρ de 0.1R a 0.9R y z de 0 a L
    rho_nodes = np.linspace(0.1 * R, 0.9 * R, N_quad)
    z_nodes = np.linspace(0, L, N_quad)
    total = 0.0

    # Cálculo del factor de decaimiento con clipping
    decay = np.array([
        np.exp(np.clip(-((alpha_n[n]**2 * D0 + eigenvalues_dict[n]) * t), -700, 700))
        for n in range(n_modes_radial)
    ])
    debug_log(f"Decay for t={t:.1e}: {decay}")

    A_nm_arr = np.array(A_nm)
    Z_eval = np.empty((N_quad, n_modes_radial, m_max))
    for j, z_val in enumerate(z_nodes):
        for n in range(n_modes_radial):
            for m in range(m_max):
                Z_eval[j, n, m] = Z_interps[n][m](z_val)
    # Usamos el factor de forma: f(ρ,z) = (2z^2 - ρ^2)/(ρ^2+z^2)^(5/2)
    for rho in rho_nodes:
        J0 = jv(0, alpha_n * rho)
        for j, z_val in enumerate(z_nodes):
            f = (2 * z_val**2 - rho**2) / (rho**2 + z_val**2 + 1e-30)**(2.5)
            sum_term = np.sum(A_nm_arr * J0[:, None] * Z_eval[j] * decay)
            if not np.isfinite(sum_term):
                sum_term = 0.0
            total += f * sum_term
    dx = (0.9 * R - 0.1 * R) / N_quad
    dz = L / N_quad
    result = (5 / (16 * np.pi)) * (total * dx * dz)
    debug_log(f"t = {t:.1e}, total*dx*dz = {total*dx*dz:e} => G(t) = {result:e}")
    return result

# Calculamos G(t) para un valor muy pequeño (aproximante a t=0)
t0 = 1e-12
G0 = compute_G_debug(t0)  # Este será nuestro valor de normalización
debug_log(f"G(t0) = {G0:e}")

# Ahora calculamos la función normalizada G_norm(t)= G(t)/G0
t_values = np.logspace(-9, -2, 25)
G_values = Parallel(n_jobs=4)(delayed(compute_G_debug)(t) for t in t_values)
G_norm = np.array(G_values) / (G0 if G0 != 0 else 1)
debug_log("Valores de G_norm(t):")
for t_val, G_val in zip(t_values, G_norm):
    msg = f"{t_val:.3e}  {G_val:.3e}"
    debug_log(msg)

plt.figure(figsize=(10, 6))
plt.loglog(t_values, G_norm, 'o-', color='navy', markersize=6,
           linewidth=1.5, label='Simulación')
plt.plot(t_values, 1e-8 * t_values**-1.5 / (G0 if G0 != 0 else 1), '--',
         color='crimson', linewidth=1.5, label=r'Teoría $t^{-3/2}$ (normalizada)')
plt.xlabel('Tiempo (s)', fontsize=12)
plt.ylabel(r'$G(t)/G(0)$', fontsize=12)
plt.title('Función de Correlación Nano-NMR Normalizada', fontsize=14)
plt.legend()
plt.grid(which='both', alpha=0.4)
plt.savefig('correlacion_nanoNMR_normalizada.pdf', bbox_inches='tight', dpi=300)
plt.close()

# ---------------------------------------------
# 9. Visualización 3D de la Densidad de Probabilidad con Depuración
# ---------------------------------------------
def plot_density_3d(t, n_points=30):
    debug_log(f"Plotting density 3D for t = {t:.1e}")
    rho_vals = np.linspace(0.2 * R, 0.8 * R, n_points)
    z_vals = np.linspace(0, L, n_points)
    Rg, Zg = np.meshgrid(rho_vals, z_vals)
    P = np.zeros_like(Rg)
    
    decay = np.array([
        np.exp(np.clip(-((alpha_n[n]**2 * D0 + eigenvalues_dict[n]) * t), -700, 700))
        for n in range(n_modes_radial)
    ])
    
    Z = np.empty((len(z_vals), n_modes_radial, m_max))
    for j, z_val in enumerate(z_vals):
        for n in range(n_modes_radial):
            for m in range(m_max):
                Z[j, n, m] = Z_interps[n][m](z_val)
    A_nm_arr = np.array(A_nm)
    for i, rho in enumerate(rho_vals):
        J0 = jv(0, alpha_n * rho)
        for j, z_val in enumerate(z_vals):
            f = (2 * z_val**2 - rho**2) / (rho**2 + z_val**2 + 1e-30)**(2.5)
            sum_term = np.sum(A_nm_arr * J0[:, None] * Z[j] * decay)
            if not np.isfinite(sum_term):
                sum_term = 0.0
            P[j, i] = np.abs(sum_term)
    
    P = np.nan_to_num(P, nan=0.0, posinf=1e40, neginf=0.0)
    dx = (0.8 * R - 0.2 * R) / n_points
    dz = L / n_points
    debug_log(f"Sumatoria (P sum): {np.sum(P)*dx*dz:e}")
    P = np.clip(P, 0, 1e8)
    
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(Rg * 1e6, Zg * 1e9, P, cmap='plasma',
                           rstride=1, cstride=1, alpha=0.9)
    ax.set_xlabel(r'$\rho$ (μm)', fontsize=12)
    ax.set_ylabel('z (nm)', fontsize=12)
    ax.set_zlabel('P(ρ,z,t)', fontsize=12)
    plt.title(f'Densidad en t = {t:.1e} s', fontsize=14)
    fig.colorbar(surf, shrink=0.6, aspect=20)
    plt.savefig(f'densidad_3d_t_{t:.0e}.pdf', bbox_inches='tight')
    plt.close()

# Visualización 3D para algunos tiempos
for t in [1e-9, 1e-6, 1e-3]:
    plot_density_3d(t)

Malla en z: min = 0.00e+00, max = 2.00e-08
alpha_n: [ 2404825.55769577  5520078.11028631  8653727.91291101 11791534.43901428
 14930917.70848779 18071063.96791092 21211636.62987926 24352471.5307493 ]


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   8 out of   8 | elapsed:   10.6s finished


A_nm[0] = [-177664.42060046   10949.41615042  -23397.28629761  -10954.00644114
   39094.75142988   73432.39236836   54103.71934789 -176317.99659979
   42831.07562165   56990.27169612]
A_nm[1] = [ 202041.42205789   49781.79126495   12315.31897285  -30242.2036179
  159882.81285265    5806.5580311  -157740.66278325  265946.80628195
  -95551.73844221  -60714.24439266]
A_nm[2] = [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
A_nm[3] = [26653.98628538 -1625.67841548   840.14668117 -2154.50622498
 13828.57815305 15912.76976259  5105.77443365 15164.71658933
 19815.42177168 -6106.53834196]
A_nm[4] = [ -24509.93907868    -242.10441931   40123.30706674   -6098.32977943
  100239.33507473 -111357.95757793    7708.14952319   33426.49326188
   77731.57727797   25257.99954595]
A_nm[5] = [  39083.77392769     810.12675771  -45217.28545161   81182.63859199
   -5749.09447462  191996.72873263   74857.52743617   51380.64405106
 -142615.95411873   23440.35756946]
A_nm[6] = [-0. -0. -0. -0. -0. -0. -0. -0. -0. -0.]
A_nm[7]

C:\Users\allan\AppData\Local\Temp\ipykernel_6788\1004686719.py:242: RuntimeWarning: overflow encountered in multiply
  sum_term = np.sum(A_nm_arr * J0[:, None] * Z[j] * decay)
C:\ProgramData\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\ProgramData\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Plotting density 3D for t = 1.0e-03
Sumatoria (P sum): inf


C:\Users\allan\AppData\Local\Temp\ipykernel_6788\1004686719.py:242: RuntimeWarning: overflow encountered in multiply
  sum_term = np.sum(A_nm_arr * J0[:, None] * Z[j] * decay)
C:\ProgramData\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\ProgramData\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
